In [1]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# Đọc các file
nodes = pd.read_csv('../Dataset/nodes.csv')
segments = pd.read_csv('../Dataset/segments.csv')
segment_status = pd.read_csv('../Dataset/segment_status.csv')
train_df = pd.read_csv('../Dataset/train.csv')

In [3]:
# Tạo edge_index: 2 x num_edges tensor
edge_index = torch.tensor([
    segments['s_node_id'].values,
    segments['e_node_id'].values
], dtype=torch.long)

# Nếu muốn làm graph undirected, thêm cả chiều ngược lại:
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)


/tmp/ipykernel_32121/2581468099.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  edge_index = torch.tensor([


In [4]:
node_features = torch.tensor(nodes[['long', 'lat']].values, dtype=torch.float)

In [5]:
# Pivot dữ liệu segment_status để mỗi segment là 1 row, mỗi updated_at là 1 cột
pivot_velocity = segment_status.pivot(index='segment_id', columns='updated_at', values='velocity')

# Fill missing data (ví dụ điền 0 hoặc dùng forward fill nếu cần)
pivot_velocity = pivot_velocity.fillna(0)

# Chuyển thành tensor
velocity_tensor = torch.tensor(pivot_velocity.values, dtype=torch.float)


In [6]:
train_data, test_data = train_test_split(velocity_tensor, test_size=0.2, random_state=42)

# Dataloader
train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
test_loader = DataLoader(test_data, batch_size=4)


In [7]:
class GRNNModel(nn.Module):
    def __init__(self, node_in_feats, gnn_hidden, rnn_hidden):
        super(GRNNModel, self).__init__()
        # --- GNN: xử lý không gian
        self.gnn = GCNConv(node_in_feats, gnn_hidden)

        # --- RNN: xử lý thời gian
        self.rnn = nn.GRU(input_size=gnn_hidden, hidden_size=rnn_hidden, batch_first=True)

        # --- Output layer
        self.fc = nn.Linear(rnn_hidden, 1)

    def forward(self, x, edge_index):
        batch_size, time_steps, num_nodes, _ = x.size()
        outputs = []

        for t in range(time_steps):
            xt = x[:, t, :, :]  # Shape: [batch_size, num_nodes, node_features]
            xt = xt.view(-1, xt.shape[-1])  # flatten batch and node dims
            gnn_out = self.gnn(xt, edge_index)  # Apply GNN
            gnn_out = gnn_out.view(batch_size, num_nodes, -1)

            gnn_out = torch.mean(gnn_out, dim=1)  # Aggregate nodes
            outputs.append(gnn_out.unsqueeze(1))

        rnn_input = torch.cat(outputs, dim=1)  # [batch_size, time_steps, gnn_hidden]
        rnn_out, _ = self.rnn(rnn_input)

        out = self.fc(rnn_out[:, -1, :])  # Predict from the last RNN output
        return out.squeeze()


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GRNNModel(node_in_feats=2, gnn_hidden=64, rnn_hidden=32).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

epochs = 100
train_losses = []

# Giả định: số bước thời gian
time_steps = train_data.shape[1]

# node_features: [num_nodes, node_dim] → cần chuẩn hóa
node_feats = node_features.to(device)

# Chuyển edge_index sang device
edge_idx = edge_index.to(device)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for batch in train_loader:
        batch = batch.to(device)  # batch: [batch_size, time_steps]

        # Chuẩn bị input cho GRNN:
        # → input_features: [batch_size, time_steps, num_nodes, node_dim]
        input_features = node_feats.unsqueeze(0).unsqueeze(0)  # [1,1,num_nodes,node_dim]
        input_features = input_features.repeat(batch.size(0), time_steps, 1, 1)  # [B,T,N,D]

        # Model dự đoán
        preds = model(input_features, edge_idx)  # [batch_size]

        # Target là giá trị velocity tại thời điểm cuối cùng
        target = batch[:, -1]  # [batch_size]

        loss = loss_fn(preds, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_loader))

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {epoch_loss / len(train_loader):.4f}")


OutOfMemoryError: CUDA out of memory. Tried to allocate 1023.53 GiB. GPU 0 has a total capacity of 3.80 GiB of which 3.69 GiB is free. Including non-PyTorch memory, this process has 104.00 MiB memory in use. Of the allocated memory 7.94 MiB is allocated by PyTorch, and 14.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        input_features = node_features.unsqueeze(0).repeat(batch.size(0), batch.size(1), 1, 1).to(device)
        preds = model(input_features, edge_index.to(device))

        y_true.append(batch[:, -1].cpu().numpy())
        y_pred.append(preds.cpu().numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)


In [ ]:
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

# Vì velocity là số thực, accuracy và f1 thường dùng khi phân loại. 
# Nếu cần, bạn có thể binarize:
y_true_bin = (y_true > y_true.mean()).astype(int)
y_pred_bin = (y_pred > y_true.mean()).astype(int)

acc = accuracy_score(y_true_bin, y_pred_bin)
f1 = f1_score(y_true_bin, y_pred_bin)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
plt.figure(figsize=(10,6))
plt.plot(y_true[:100], label='Ground Truth', marker='o')
plt.plot(y_pred[:100], label='Prediction', marker='x')
plt.legend()
plt.title('Comparison between Real and Predicted values')
plt.xlabel('Sample')
plt.ylabel('Velocity or LOS')
plt.grid(True)
plt.show()
